# 🎵 ACE-Step v1.5 **base** — Entraînement LoRA / LoKR avec Side-Step (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/walidb27/ACE-Step-1.5/blob/claude/nice-volta-wvnuc6/notebooks/ACE_Step_v15_Base_SideStep_Colab.ipynb)

L'interface Gradio d'ACE-Step n'entraîne que le modèle **turbo** (timesteps discrets « shift=3 »).
Ce notebook utilise **Side-Step** (`train.py`, mode `fixed`, intégré au dépôt) qui reproduit le
vrai schéma d'entraînement des modèles **base / sft** :

- timesteps **continus** (logit-normal, paramètres lus dans le `config.json` du modèle) ;
- **CFG dropout** (le modèle base a été entraîné avec la guidance sans classifieur) ;
- LoRA (PEFT) ou LoKR (LyCORIS), gradient checkpointing, optimiseurs 8-bit / Prodigy.

**Étapes :** 1) GPU & Drive → 2) installation → 3) téléchargement des poids → 4) dataset →
5) prétraitement → 6) entraînement → 7) récupération de l'adaptateur.

> 💡 GPU conseillé : **L4 ou A100** (bf16). Le T4 fonctionne (fp16, ~7 Go de VRAM avec gradient
> checkpointing) mais il est plus lent et le fp16 est moins stable.

## 1. GPU et Google Drive

In [ ]:
#@title Vérifier le GPU et monter Google Drive
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Dossier de travail sur le Drive (dataset, tenseurs, sorties). Tout y est conservé
#@markdown si la session Colab se coupe.
DRIVE_WORKDIR = "/content/drive/MyDrive/ace_step_training"  #@param {type:"string"}

import os
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "⚠️ Aucun GPU détecté : Exécution > Modifier le type d'exécution > GPU")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_WORKDIR
else:
    WORKDIR = "/content/ace_step_training"
os.makedirs(WORKDIR, exist_ok=True)
print("Dossier de travail :", WORKDIR)

## 2. Installation (ACE-Step + Side-Step)

In [ ]:
#@title Cloner le dépôt et installer les dépendances (~5 min)
REPO_URL = "https://github.com/walidb27/ACE-Step-1.5.git"  #@param {type:"string"}
REPO_BRANCH = "claude/nice-volta-wvnuc6"  #@param {type:"string"}
REPO_DIR = "/content/ACE-Step-1.5"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# requirements.txt du dépôt, sans flash-attn (compilation de plusieurs dizaines de minutes ;
# Side-Step bascule automatiquement sur SDPA).
!grep -v -i "flash-attn" requirements.txt > /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
# Dépendances Side-Step (CLI) + optimiseurs optionnels (AdamW8bit, Prodigy)
!pip install -q -r requirements-sidestep.txt bitsandbytes prodigyopt

!python -c "import torch, torchaudio; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| bf16', torch.cuda.is_available() and torch.cuda.is_bf16_supported())"

## 3. Télécharger les poids

On ne télécharge que ce qui sert à l'entraînement : **VAE**, encodeur de texte
**Qwen3-Embedding-0.6B**, `silence_latent.pt` et le DiT **acestep-v15-base**
(le LM 1.7B et le DiT turbo ne sont pas nécessaires).

In [ ]:
#@title Télécharger VAE + encodeur texte + modèle base
#@markdown Mettre les checkpoints sur le Drive évite de les retélécharger (~6 Go) à chaque session.
CHECKPOINTS_ON_DRIVE = False  #@param {type:"boolean"}
HF_TOKEN = ""  #@param {type:"string"}

import os
from huggingface_hub import snapshot_download

CKPT_DIR = os.path.join(WORKDIR, "checkpoints") if CHECKPOINTS_ON_DRIVE else "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
token = HF_TOKEN or None

# Composants partagés depuis le dépôt principal (sans le LM ni le DiT turbo)
snapshot_download(
    repo_id="ACE-Step/Ace-Step1.5",
    local_dir=CKPT_DIR,
    allow_patterns=["vae/*", "Qwen3-Embedding-0.6B/*", "acestep-v15-turbo/silence_latent.pt"],
    token=token,
)

# DiT base (le downloader du dépôt synchronise aussi le code du modèle)
cmd = f"python -m acestep.model_downloader --model acestep-v15-base --skip-main --dir {CKPT_DIR}"
if token:
    cmd += f" --token {token}"
!{cmd}

!ls -la {CKPT_DIR} {CKPT_DIR}/acestep-v15-base

## 4. Dataset

Placez vos fichiers audio (`.wav .mp3 .flac .ogg .opus .m4a`, idéalement 30 s à 4 min, **5 morceaux
minimum**) dans `AUDIO_DIR` (sur le Drive) ou envoyez un `.zip` avec la cellule ci-dessous.

**Métadonnées (recommandé)** : un JSON au format Side-Step / Dataset Builder d'ACE-Step
(`caption`, `genre`, `lyrics`, `bpm`, `keyscale`, `custom_tag`…). Le JSON exporté par l'onglet
*Dataset* de l'interface Gradio fonctionne directement. Sans JSON, la légende est déduite du nom de
fichier et les morceaux sont considérés comme instrumentaux.
Voir `docs/sidestep/Dataset Preparation.md`.

In [ ]:
#@title Choisir le dossier audio (et éventuellement le JSON)
AUDIO_DIR = ""  #@param {type:"string"}
#@markdown Laisser vide = `<WORKDIR>/audio`
DATASET_JSON = ""  #@param {type:"string"}
#@markdown Optionnel. Les chemins relatifs dans le JSON sont résolus depuis le dossier du JSON.
UPLOAD_ZIP = False  #@param {type:"boolean"}
#@markdown Cocher pour envoyer un .zip d'audios depuis votre ordinateur.

import os
import zipfile

AUDIO_DIR = AUDIO_DIR or os.path.join(WORKDIR, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

if UPLOAD_ZIP:
    from google.colab import files
    for name in files.upload():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as zf:
                zf.extractall(AUDIO_DIR)
            os.remove(name)

EXTS = (".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a")
found = [os.path.join(r, f) for r, _, fs in os.walk(AUDIO_DIR) for f in fs if f.lower().endswith(EXTS)]
print(f"{len(found)} fichier(s) audio dans {AUDIO_DIR}")
for f in found[:20]:
    print(" -", os.path.relpath(f, AUDIO_DIR))
if DATASET_JSON and not os.path.isfile(DATASET_JSON):
    raise FileNotFoundError(f"JSON introuvable : {DATASET_JSON}")
if not found and not DATASET_JSON:
    print("⚠️ Aucun audio trouvé : ajoutez des fichiers avant le prétraitement.")

## 5. Prétraitement (audio → tenseurs `.pt`)

Deux passes à faible VRAM : VAE + encodeur texte, puis encodeur de conditionnement du DiT **base**.
Les tenseurs doivent être produits **avec le même modèle que celui qu'on entraîne** (ici `base`).
À ne refaire que si le dataset change.

In [ ]:
#@title Lancer le prétraitement
DATASET_NAME = "my_dataset"  #@param {type:"string"}
MAX_DURATION = 240  #@param {type:"number"}
FORCE_REPROCESS = False  #@param {type:"boolean"}

%cd /content/ACE-Step-1.5
import os
import shlex
import shutil

TENSOR_DIR = os.path.join(WORKDIR, "tensors", DATASET_NAME)
if FORCE_REPROCESS and os.path.isdir(TENSOR_DIR):
    shutil.rmtree(TENSOR_DIR)

if os.path.isfile(os.path.join(TENSOR_DIR, "manifest.json")):
    print("Tenseurs déjà présents :", TENSOR_DIR, "(cocher FORCE_REPROCESS pour refaire)")
else:
    args = [
        "python", "-m", "acestep.training_v2.cli.train_fixed", "--plain",
        "--checkpoint-dir", CKPT_DIR,
        "--model-variant", "base",
        "--preprocess",
        "--audio-dir", AUDIO_DIR,
        "--tensor-output", TENSOR_DIR,
        "--max-duration", str(MAX_DURATION),
    ]
    if DATASET_JSON:
        args += ["--dataset-json", DATASET_JSON]
    cmd = shlex.join(args)
    print(cmd)
    !{cmd}

!ls {TENSOR_DIR} | head -20

## 6. Entraînement Side-Step sur le modèle base

Valeurs par défaut = preset `recommended` de Side-Step. `shift=1.0` et `50` steps sont enregistrés
comme métadonnées d'inférence (valeurs correctes pour base) ; ils n'influencent pas l'entraînement.

Repères :
- **VRAM juste ?** `OFFLOAD_ENCODER`, `OPTIMIZER = adamw8bit`, rang plus bas.
- **Sur-apprentissage** (la sortie recopie le dataset) : moins d'époques, rang plus bas, plus de données.
- **Pas d'effet audible** : plus d'époques ou rang plus élevé.

In [ ]:
#@title Paramètres d'entraînement
RUN_NAME = "my_lora_base"  #@param {type:"string"}
ADAPTER_TYPE = "lora"  #@param ["lora", "lokr"]
RANK = 64  #@param {type:"integer"}
ALPHA = 128  #@param {type:"integer"}
DROPOUT = 0.1  #@param {type:"number"}
ATTENTION_TYPE = "both"  #@param ["both", "self", "cross"]
LEARNING_RATE = 1e-4  #@param {type:"number"}
EPOCHS = 100  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
GRADIENT_ACCUMULATION = 4  #@param {type:"integer"}
WARMUP_STEPS = 100  #@param {type:"integer"}
OPTIMIZER = "adamw"  #@param ["adamw", "adamw8bit", "adafactor", "prodigy"]
SCHEDULER = "cosine"  #@param ["cosine", "cosine_restarts", "linear", "constant", "constant_with_warmup"]
CFG_RATIO = 0.15  #@param {type:"number"}
SAVE_EVERY = 10  #@param {type:"integer"}
PRECISION = "auto"  #@param ["auto", "bf16", "fp16", "fp32"]
OFFLOAD_ENCODER = False  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}
#@markdown ---
#@markdown **Reprise** : chemin d'un checkpoint, ex. `<WORKDIR>/output/my_lora_base/checkpoints/epoch_50`
RESUME_FROM = ""  #@param {type:"string"}

import os
import shlex

OUTPUT_DIR = os.path.join(WORKDIR, "output", RUN_NAME)
if OPTIMIZER == "prodigy" and LEARNING_RATE < 0.1:
    print("ℹ️ Prodigy ajuste le LR lui-même : un LR de 1.0 est conseillé.")

train_args = [
    "python", "train.py", "--plain", "--yes", "fixed",
    "--checkpoint-dir", CKPT_DIR,
    "--model-variant", "base",
    "--dataset-dir", TENSOR_DIR,
    "--output-dir", OUTPUT_DIR,
    "--adapter-type", ADAPTER_TYPE,
    "--attention-type", ATTENTION_TYPE,
    "--lr", str(LEARNING_RATE),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
    "--warmup-steps", str(WARMUP_STEPS),
    "--optimizer-type", OPTIMIZER,
    "--scheduler-type", SCHEDULER,
    "--cfg-ratio", str(CFG_RATIO),
    "--save-every", str(SAVE_EVERY),
    "--precision", PRECISION,
    "--seed", str(SEED),
    "--shift", "1.0",
    "--num-inference-steps", "50",
]
if ADAPTER_TYPE == "lora":
    train_args += ["--rank", str(RANK), "--alpha", str(ALPHA), "--dropout", str(DROPOUT)]
else:
    train_args += ["--lokr-linear-dim", str(RANK), "--lokr-linear-alpha", str(ALPHA)]
if OFFLOAD_ENCODER:
    train_args.append("--offload-encoder")
if RESUME_FROM:
    train_args += ["--resume-from", RESUME_FROM]

TRAIN_CMD = shlex.join(train_args)
print(TRAIN_CMD)

In [ ]:
#@title (Optionnel) TensorBoard — lancer avant l'entraînement pour suivre la loss
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/runs

In [ ]:
#@title 🚀 Lancer l'entraînement
%cd /content/ACE-Step-1.5
!{TRAIN_CMD}

## 7. Récupérer et utiliser l'adaptateur

- Adaptateur final : `<WORKDIR>/output/<RUN_NAME>/final/`
- Checkpoints intermédiaires (aussi utilisables en inférence) : `.../checkpoints/epoch_N/`

**Dans ACE-Step (Gradio)** : charger le modèle **acestep-v15-base** (pas turbo), puis
*LoRA Adapter* → chemin du dossier `final` → *Load LoRA* → *Use LoRA*.
Générer avec **shift = 1.0** et **~50 steps** ; inclure votre `custom_tag` dans le prompt.
Les adaptateurs **LoKR** ont des limitations au chargement : voir `docs/sidestep/Using Your Adapter.md`.

In [ ]:
#@title Zipper l'adaptateur final (et le télécharger)
DOWNLOAD_ZIP = True  #@param {type:"boolean"}

import os
import shutil

final_dir = os.path.join(OUTPUT_DIR, "final")
if not os.path.isdir(final_dir):
    print("Pas encore de dossier final :", final_dir)
else:
    archive = shutil.make_archive(os.path.join(WORKDIR, f"{RUN_NAME}_final"), "zip", final_dir)
    print("Archive :", archive)
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(archive)